# The multilayer perceptron

In the previous exercises, we trained a single neuron to perform linear regression and built an autodiff engine. In this exercise, we tackle a problem that a single neuron **cannot** solve: the XOR function. This will motivate the need for multiple layers and nonlinear activation functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from variable import Variable

## The XOR problem

The XOR (exclusive or) function takes two binary inputs and returns 1 if exactly one of them is 1:

| A | B | XOR |
|---|---|-----|
| 0 | 0 |  0  |
| 0 | 1 |  1  |
| 1 | 0 |  1  |
| 1 | 1 |  0  |

If you plot these four points with their labels, you'll see that no single straight line can separate the 0s from the 1s.

In [ ]:
def xor():
    """Generate the XOR dataset."""
    A_B = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    XOR = np.array([0, 1, 1, 0])
    return A_B, XOR


A_B, XOR = xor()
plt.scatter(A_B[:, 0], A_B[:, 1], c=XOR, cmap="coolwarm", s=100, edgecolors="k")
plt.xlabel("A")
plt.ylabel("B")
plt.title("XOR: no single line can separate the classes")
plt.show()

## Exercise 1: A single perceptron

We first implement a single 2D perceptron with a binary threshold activation to confirm that it **cannot** solve XOR.

The perceptron computes:

$$
z = w_1 A + w_2 B + b, \qquad \hat{y} = \begin{cases} 1 & z > T \\ 0 & z \leq T \end{cases}
$$

We use the `Variable` class from the previous exercise for automatic gradient computation.

**Fill in** the missing code in `forward`, `evaluate`, and `gradient_step`.

In [ ]:
class Perceptron2D:
    def __init__(self, T=0, lr=1e-1):
        """
        A 2D perceptron with binary threshold activation.

        Parameters:
            T: Threshold for the binary activation
            lr: Learning rate for gradient descent
        """
        self.T = T
        self.lr = lr
        self.w1 = Variable(np.random.uniform())
        self.w2 = Variable(np.random.uniform())
        self.bias = Variable(-1)

    def forward(self, A, B):
        """Compute the linear combination z = w1*A + w2*B + bias."""
        y_hat = ...  # TODO
        return y_hat

    def evaluate(self, A, B):
        """Apply the threshold to produce a binary prediction."""
        y_hat = ...  # TODO: call forward
        # TODO: threshold at self.T (use y_hat.value since > isn't defined for Variable)
        return ...

    def loss_fn(self, A, B, y):
        """Squared error loss."""
        y_hat = self.forward(A, B)
        return (y_hat - y) ** 2

    def gradient_step(self, A, B, y):
        """Compute loss, get gradients via autodiff, and update weights."""
        loss = ...  # TODO: compute loss
        grads = ...  # TODO: get gradients from loss
        # Update weights (not the bias)
        w1 = self.w1 - self.lr * grads[self.w1]
        w2 = self.w2 - self.lr * grads[self.w2]
        # Re-wrap as Variable to avoid infinite recursion in the graph
        self.w1 = Variable(w1.value)
        self.w2 = Variable(w2.value)
        return loss.value

## Exercise 2: Train the perceptron

**Fill in** the training loop below: for each epoch, loop over the data points and call `gradient_step`.

In [ ]:
def train(model, A_B, XOR, epochs=1000):
    """Train the model for a given number of epochs."""
    loss_history = []
    for epoch in range(epochs):
        loss = 0
        for i in range(len(XOR)):
            A, B = A_B[i]
            y = XOR[i]
            # TODO: do a gradient step and accumulate the loss
            loss += ...
        loss_history.append(loss)
    return loss_history

In [ ]:
model = Perceptron2D()
A_B, XOR = xor()
loss_history = train(model, A_B, XOR, epochs=1000)

plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Single perceptron on XOR")
plt.show()

## Decision boundaries

Let's visualize the decision boundary to understand what the perceptron has learned. The code below evaluates the model on a 2D grid and plots the result.

In [ ]:
def plot_decision_boundary(model, A_B, XOR, fig=None, ax=None):
    """Plot the decision boundary of a model on the XOR data."""
    x = np.linspace(-0.2, 1.2, 100)
    y = np.linspace(-0.2, 1.2, 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    for i in range(len(X)):
        for j in range(len(X[0])):
            Z[i, j] = model.evaluate(Variable(X[i, j]), Variable(Y[i, j]))
    if fig is None or ax is None:
        fig, ax = plt.subplots()
    ax.contourf(X, Y, Z, alpha=0.2)
    ax.scatter(A_B[:, 0], A_B[:, 1], c=XOR, cmap="coolwarm", s=100, edgecolors="k")
    ax.set_xlabel("A")
    ax.set_ylabel("B")
    return fig, ax


plot_decision_boundary(model, A_B, XOR)
plt.title("Single perceptron decision boundary")
plt.show()

The decision boundary is a straight line — confirming that a single perceptron **cannot** solve XOR. No matter how long we train, the loss will not go to zero.

## Exercise 3: Why do we need nonlinearity? (Optional)

Before adding more layers, let's understand why simply stacking linear layers doesn't help.

### a) Prove that stacking linear layers is equivalent to a single linear layer

Consider a network with $n$ layers, where each layer is a linear function:

$$ f_i(z) = W_i z + b_i $$

The full network is the composition $f(x) = f_n \circ f_{n-1} \circ \ldots \circ f_1(x)$.

Your task: Show that $f(x)$ can always be rewritten as a single linear transformation $f(x) = W'x + b'$, no matter how many layers we stack.

Step-by-step hint:

1. Start with just two layers. Write out $f_2(f_1(x))$ by substituting $f_1(x) = W_1 x + b_1$ into $f_2(z) = W_2 z + b_2$:

$$ f_2(f_1(x)) = W_2(W_1 x + b_1) + b_2 $$

2. Expand and collect terms into the form $W'x + b'$.
3. Argue why this pattern extends to any number of layers.

*Write your proof here (or on paper).*

### b) Polynomial activation functions

Now consider what happens if we use a polynomial activation function instead of a linear one.

1. Take a two-layer network where each layer applies $f_i(z) = W_i z^2$ (no bias). Write out $f_2(f_1(x))$ and simplify. What degree polynomial is the result?

2. Can a single-layer network with the right polynomial degree represent the same function? What does this tell you about the expressiveness gained by stacking polynomial layers?

Hint for part 1: substitute $f_1(x) = W_1 x^2$ into $f_2(z) = W_2 z^2$.

*Write your answer here.*

## The sigmoid activation

The binary threshold used in the perceptron is not differentiable, which limits training to simple gradient tricks. The **sigmoid** function is a smooth, differentiable replacement:

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

It maps any real number to the range $(0, 1)$, producing a smooth "S-curve" that approximates the step function.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


x = np.linspace(-5, 5, 100)
plt.plot(x, sigmoid(x), label=r"$\sigma(x)$")
plt.plot(x, (x > 0).astype(float), "--", label="Step function", alpha=0.5)
plt.xlabel("x")
plt.ylabel(r"$\sigma(x)$")
plt.legend()
plt.title("Sigmoid vs step function")
plt.show()

## XOR as a composition of linearly separable functions

Although XOR is not linearly separable, it can be decomposed into functions that *are*:

$$
\text{XOR}(A, B) = \text{OR}(A, B) \cdot \text{NAND}(A, B)
$$

Since OR, NAND, and AND are each linearly separable, a two-layer network with two neurons in the hidden layer can solve XOR.

In [ ]:
A, B = A_B.T
fig, ax = plt.subplots(ncols=3, figsize=(12, 4))
ax[0].scatter(A, B, c=A | B, cmap="bwr", s=100, edgecolors="k")
ax[0].set_title("OR")
ax[1].scatter(A, B, c=~(A & B), cmap="bwr", s=100, edgecolors="k")
ax[1].set_title("NAND")
ax[2].scatter(A, B, c=A ^ B, cmap="bwr", s=100, edgecolors="k")
ax[2].set_title("XOR = OR · NAND")
for a in ax:
    a.set_xlabel("A")
    a.set_ylabel("B")
plt.tight_layout()
plt.show()

## Exercise 4: A two-layer MLP

Now implement a two-layer MLP with sigmoid activations. The architecture:

- **Hidden layer**: 2 neurons, each computing $\sigma(w_1 A + w_2 B + b)$
- **Output layer**: 1 neuron, computing $\sigma(w_1 h_1 + w_2 h_2 + b)$

**Fill in** the `forward` method. The rest of the class (loss, gradient step) works the same way as the single perceptron.

In [ ]:
from variable import exp


class MLP:
    def __init__(self, T=0.5, lr=1e-1):
        """
        A two-layer perceptron with 2 hidden neurons and 1 output neuron.
        Uses sigmoid activation.
        """
        self.T, self.lr = T, lr
        # Hidden layer: neuron 1
        self.w11 = Variable(np.random.normal(scale=0.1))
        self.w12 = Variable(np.random.normal(scale=0.1))
        self.b11 = Variable(np.random.normal(scale=0.01))
        # Hidden layer: neuron 2
        self.w13 = Variable(np.random.normal(scale=0.1))
        self.w14 = Variable(np.random.normal(scale=0.1))
        self.b12 = Variable(np.random.normal(scale=0.01))
        # Output layer
        self.w21 = Variable(np.random.normal(scale=0.1))
        self.w22 = Variable(np.random.normal(scale=0.1))
        self.b21 = Variable(np.random.normal(scale=0.01))

    def _sigmoid(self, x):
        """Sigmoid using the Variable class."""
        return 1 / (1 + exp(-x))

    def forward(self, A, B):
        """
        Forward pass of the MLP.

        TODO: Compute the output of each hidden neuron, apply sigmoid,
        then compute the output neuron and apply sigmoid again.
        """
        # Hidden layer
        # z1 = ...  (linear combination for hidden neuron 1)
        # z2 = ...  (linear combination for hidden neuron 2)
        # h1 = ...  (sigmoid of z1)
        # h2 = ...  (sigmoid of z2)
        # Output layer
        # z_out = ... (linear combination of h1, h2)
        # return sigmoid(z_out)
        raise NotImplementedError

    def evaluate(self, A, B):
        """Threshold the forward pass for binary classification."""
        y_hat = self.forward(A, B)
        return y_hat.value > self.T

    def loss_fn(self, A, B, y):
        """Squared error loss."""
        y_hat = self.forward(A, B)
        return (y_hat - y) ** 2

    def gradient_step(self, A, B, y):
        """Compute loss, backpropagate, and update all weights."""
        loss = self.loss_fn(A, B, y)
        grads = loss.gradients
        # Update all weights and biases
        self.w11 = Variable((self.w11 - self.lr * grads[self.w11]).value)
        self.w12 = Variable((self.w12 - self.lr * grads[self.w12]).value)
        self.w13 = Variable((self.w13 - self.lr * grads[self.w13]).value)
        self.w14 = Variable((self.w14 - self.lr * grads[self.w14]).value)
        self.b11 = Variable((self.b11 - self.lr * grads[self.b11]).value)
        self.b12 = Variable((self.b12 - self.lr * grads[self.b12]).value)
        self.w21 = Variable((self.w21 - self.lr * grads[self.w21]).value)
        self.w22 = Variable((self.w22 - self.lr * grads[self.w22]).value)
        self.b21 = Variable((self.b21 - self.lr * grads[self.b21]).value)
        return loss.value

In [ ]:
# Train and visualize
model = MLP(lr=1e-1)
loss_history = train(model, A_B, XOR, epochs=10000)

fig, axs = plt.subplots(ncols=2, figsize=(10, 4))
axs[0].plot(loss_history)
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Loss")
axs[0].set_title("MLP training loss")

plot_decision_boundary(model, A_B, XOR, fig=fig, ax=axs[1])
axs[1].set_title("MLP decision boundary")
plt.tight_layout()
plt.show()

The MLP should produce a **nonlinear** decision boundary that correctly separates all four XOR data points — something the single perceptron could never do.